# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

print(f"Dataset Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets defined by their `@id` in the dataset's metadata
record_sets = dataset.get_record_sets()
print("Available Record Sets (@id):")
for rs in record_sets:
    print(f"- {rs['@id']} (name: {rs.get('name', 'N/A')})")

# For each record set, list its fields (columns) and their @id
overview = {}
for rs in record_sets:
    fid_list = []
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for f in fields:
        fid_list.append(f["@id"] if isinstance(f, dict) and '@id' in f else f)
    overview[rs['@id']] = fid_list
    print(f"  - Fields for {rs['@id']}: {fid_list if fid_list else 'N/A'}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Prepare the list of record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}

# Load each record set as a pandas DataFrame
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)

# Print out columns from the first available record set
example_rs_id = None
for rs_id, df in dataframes.items():
    example_rs_id = rs_id
    print(f"Columns in {rs_id}: {df.columns.tolist()}")
    display(df.head())
    break
# Save example_rs_id for later use

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section applies operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

# If you know the name of a numeric field, set it here. We'll try to guess if not.
if example_rs_id is not None:
    df = dataframes[example_rs_id]
    # Pick a numeric field (try to find a float/integer field)
    num_col = None
    for col in df.columns:
        # Try converting a column to numeric; pick first that works
        if pd.api.types.is_numeric_dtype(df[col]):
            num_col = col
            break
        try:
            df_tmp = pd.to_numeric(df[col])
            num_col = col
            df[col] = df_tmp
            break
        except Exception:
            continue
    if num_col is None:
        raise ValueError('No numeric field found in the records set! Please adjust field selection.')
    print(f"Using '{num_col}' as numeric field.")

    # Filter records with value above threshold
    threshold = df[num_col].mean() if df[num_col].dtype != object else 0
    filtered_df = df[df[num_col] > threshold]
    print(f"Filtered records in field '{num_col}' > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{num_col}_normalized"] = (filtered_df[num_col] - filtered_df[num_col].mean()) / filtered_df[num_col].std()
    print(f"Normalized '{num_col}' for filtered records:")
    display(filtered_df[[num_col, f"{num_col}_normalized"]].head())

    # Try grouping by a categorical field if possible
    group_field = None
    for col in df.columns:
        if col != num_col and not pd.api.types.is_numeric_dtype(df[col]):
            group_field = col
            break
    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped statistics by '{group_field}':")
        display(grouped_df.head())
    else:
        print("No suitable categorical group field found to group by.")
else:
    print("No record sets with dataframes could be loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if example_rs_id is not None and num_col is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[num_col].dropna(), kde=True)
    plt.xlabel(num_col)
    plt.title(f"Distribution of {num_col}")
    plt.show()

    # If there is a group field, visualize grouped means
    if group_field is not None:
        plt.figure(figsize=(8,4))
        sns.barplot(
            x=grouped_df.index,
            y=grouped_df[num_col]
        )
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {num_col}")
        plt.title(f"Mean {num_col} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Visualization skipped: No numeric field available.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Using the FAIR² Croissant dataset describing mass spectrometry of human mast cell EVs, we successfully loaded metadata and records using `mlcroissant`.
- The dataset provides rich structured fields, including protein and peptide quantitative results.
- Basic EDA and visualizations enable exploration of numeric field distributions and summary statistics by group.
- The approach can be extended to deeper domain-specific analysis and downstream ML workflows.